# Práctica 07 — Tests de hipótesis I

**Curso:** Estadística Aplicada con Python · **Clase 07** · **Tiempo estimado:** 60-90 minutos
**Notebook de clase:** `notebooks_clase/clase07_tests_hipotesis_i.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una celda con `# TODO`: complétala.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los ejercicios con simulación usan tolerancias basadas en el error estándar, así que un resultado correcto no debería fallar por azar.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Todos los datos son sintéticos: *ejemplos ilustrativos*.

---
## Celda 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

## Ejercicio 1 — El t de una muestra, a mano

El objetivo de un servicio es una media de 3,0 s por consulta. Se miden 10 consultas (*ejemplo ilustrativo*). H0: μ = 3,0; H1: μ ≠ 3,0. Calcula:

- `media`, `s` (desviación típica muestral con `ddof=1`),
- `t_obs = (media − mu0) / (s / √n)` y sus grados de libertad `gl`,
- `p_valor` a dos colas con `stats.t.sf`,
- `rechaza_h0`: `True` si rechazas H0 con α = 0,05.

In [ ]:
tiempos = np.array([2.7, 3.1, 2.5, 3.0, 2.8, 2.6, 3.2, 2.9, 2.4, 2.8])
mu0 = 3.0
n = len(tiempos)

media = None        # TODO
s = None            # TODO
t_obs = None        # TODO
gl = None           # TODO
p_valor = None      # TODO: dos colas
rechaza_h0 = None   # TODO

# Comprobación automática
assert all(v is not None for v in (media, s, t_obs, gl, p_valor, rechaza_h0)), "Completa las líneas marcadas con TODO"
ref = stats.ttest_1samp(tiempos, mu0)
assert np.isclose(t_obs, ref.statistic), "t = (media − mu0) / (s / raíz de n), con s calculada con ddof=1"
assert gl == n - 1, "los grados de libertad son n − 1"
assert np.isclose(p_valor, ref.pvalue), "p = 2 * stats.t.sf(abs(t), gl)"
assert rechaza_h0 == (ref.pvalue < 0.05), "compara el p-value con α = 0,05"
print(f"t = {t_obs:.3f}   gl = {gl}   p = {p_valor:.4f}   ¿rechazo H0? {rechaza_h0}")
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Dos grupos independientes: Welch, efecto e IC

Comparas el tiempo de respuesta de dos servidores con tamaños y variabilidad distintos (*ejemplo ilustrativo*). Calcula:

- `res`: el resultado de `stats.ttest_ind` con **Welch** (`equal_var=False`),
- `dif`: media de `grupo_a` menos media de `grupo_b`,
- `d`: la d de Cohen (diferencia de medias / desviación típica combinada),
- `ic`: el intervalo de confianza al 95 % (`res.confidence_interval()`),
- `significativo`: `True` si p < 0,05.

In [ ]:
rng_e2 = np.random.default_rng(2026)
grupo_a = rng_e2.normal(3.0, 0.8, 30)
grupo_b = rng_e2.normal(3.5, 1.2, 35)

res = None            # TODO: test de Welch
dif = None            # TODO
d = None              # TODO: d de Cohen con desviación típica combinada
ic = None             # TODO
significativo = None  # TODO

# Comprobación automática
assert all(v is not None for v in (res, dif, d, ic, significativo)), "Completa las líneas marcadas con TODO"
assert np.isclose(res.statistic, -3.0697, atol=1e-3), "usa equal_var=False (Welch)"
assert np.isclose(dif, grupo_a.mean() - grupo_b.mean()), "dif = media de A − media de B"
assert np.isclose(d, -0.7266, atol=1e-3), "d = dif / desviación típica combinada, sp = raíz(((na−1)·va + (nb−1)·vb) / (na+nb−2))"
assert ic.low < dif < ic.high, "el IC debe contener la diferencia observada"
assert significativo is True or significativo == True, "compara el p-value con 0,05"
print(f"dif = {dif:.3f} s   IC 95 % = ({ic.low:.3f}, {ic.high:.3f})   d = {d:.2f}   p = {res.pvalue:.4f}")
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — ¿Pareado o independiente?

Se miden 20 páginas antes y después de una optimización (*ejemplo ilustrativo*): cada página tiene dos mediciones. Calcula:

- `p_pareado`: p-value del test pareado (`ttest_rel`),
- `p_indep`: p-value del test de Welch **tratando los datos como independientes** (solo para ver qué ocurre),
- `usar_pareado`: `True` si el test pareado es el adecuado para estos datos, `False` si no.

Después de los asserts, reflexiona: ¿por qué difieren tanto los dos p-values?

In [ ]:
rng_e3 = np.random.default_rng(77)
antes = rng_e3.lognormal(1.0, 0.5, 20)
despues = antes - 0.25 + rng_e3.normal(0, 0.25, 20)

p_pareado = None      # TODO
p_indep = None        # TODO
usar_pareado = None   # TODO

# Comprobación automática
assert all(v is not None for v in (p_pareado, p_indep, usar_pareado)), "Completa las líneas marcadas con TODO"
assert np.isclose(p_pareado, 3.8e-5, rtol=0.05), "usa stats.ttest_rel(antes, despues)"
assert np.isclose(p_indep, 0.656, atol=0.005), "usa stats.ttest_ind(..., equal_var=False)"
assert usar_pareado == True, "cada página tiene su pareja antes/después: los datos son pareados"
print(f"p pareado = {p_pareado:.2e}   p independiente = {p_indep:.3f}")
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Test de dos proporciones

Un test de dos variantes de un formulario (*ejemplo ilustrativo*): la variante nueva convierte 90 de 1 200 visitas y la antigua 66 de 1 200. Calcula:

- `z`, `p`: estadístico y p-value de `proportions_ztest` (pon primero la variante nueva),
- `dif`: diferencia de proporciones (nueva − antigua),
- `ic`: tupla `(inferior, superior)` con el IC al 95 % de la diferencia (error estándar **sin combinar**, con 1,96),
- `rechaza_h0`: `True` si p < 0,05.

In [ ]:
conv = np.array([90, 66])         # nueva, antigua
vis = np.array([1200, 1200])

z = None            # TODO
p = None            # TODO
dif = None          # TODO
ic = None           # TODO: (inferior, superior)
rechaza_h0 = None   # TODO

# Comprobación automática
assert all(v is not None for v in (z, p, dif, ic, rechaza_h0)), "Completa las líneas marcadas con TODO"
assert np.isclose(z, 1.9872, atol=1e-3), "usa proportions_ztest(conv, vis) con la variante nueva primero"
assert np.isclose(p, 0.0469, atol=1e-3), "el p-value es el de dos colas que devuelve proportions_ztest"
assert np.isclose(dif, 0.02), "dif = 90/1200 − 66/1200"
assert ic[0] < dif < ic[1], "el IC debe contener la diferencia"
assert np.isclose(ic[0], 0.0003, atol=5e-4) and np.isclose(ic[1], 0.0397, atol=5e-4), "IC = dif ± 1,96 · raíz(p1(1−p1)/n1 + p2(1−p2)/n2)"
assert rechaza_h0 == True, "p < 0,05"
print(f"z = {z:.3f}   p = {p:.4f}   dif = {dif:.4f}   IC 95 % = ({ic[0]:.4f}, {ic[1]:.4f})")
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Simular la tasa de falsos positivos y la potencia

Con simulación (semilla fija `rng_ej`), estima:

- `tasa_fp`: la fracción de tests de Welch con p < **0,01** cuando **H0 es cierta** (dos grupos de 25, ambos N(0, 1)), con 5 000 repeticiones. Debería ser cercana a 0,01.
- `potencia`: la fracción de tests con p < 0,05 cuando hay un efecto real de 0,8 (grupo A ~ N(0, 1), grupo B ~ N(0,8, 1)), 25 por grupo y 5 000 repeticiones.

Pista: `stats.ttest_ind(x, y, axis=1, equal_var=False)` calcula un test por cada fila.

In [ ]:
rng_ej = np.random.default_rng(11)
N = 5_000

tasa_fp = None    # TODO: H0 cierta, α = 0,01
potencia = None   # TODO: efecto real 0,8, α = 0,05

# Comprobación automática
assert tasa_fp is not None and potencia is not None, "Completa las líneas marcadas con TODO"
ee_fp = np.sqrt(0.01 * 0.99 / N)
assert abs(tasa_fp - 0.01) < 4 * ee_fp, "con H0 cierta, la tasa de p < α debe ser ≈ α (tolerancia: 4 errores estándar)"
assert abs(potencia - 0.79) < 0.04, "con efecto 0,8 y n = 25 por grupo la potencia teórica es ≈ 0,79"
print(f"tasa de falsos positivos (α = 0,01) = {tasa_fp:.4f}   potencia = {potencia:.3f}")
print("Ejercicio 5 correcto ✔")

---
## Reto final — Un informe honesto

Un equipo prueba un cambio en un servidor con solo 12 mediciones por grupo (*ejemplo ilustrativo*). Completa la función `informe_comparacion(a, b)`, que devuelve un diccionario con las claves `"dif"` (media de a − media de b), `"ic_inf"` y `"ic_sup"` (IC al 95 % de la diferencia con Welch), `"t"`, `"p"` y `"d"` (d de Cohen).

Después, elige en `respuesta` la única interpretación correcta del resultado:

- `"a"`: no hay ninguna diferencia real entre los dos grupos.
- `"b"`: no hay evidencia suficiente para afirmar una diferencia, pero el IC es ancho y no descarta efectos relevantes; con tan pocos datos el test tiene poca potencia.
- `"c"`: H0 es cierta con un 95 % de probabilidad.

In [ ]:
def informe_comparacion(a, b):
    """Devuelve dif, IC 95 % (Welch), t, p y d de Cohen."""
    # TODO: test de Welch, diferencia, IC y d de Cohen
    return None

rng_r = np.random.default_rng(910)
grupo_1 = rng_r.normal(3.0, 1.0, 12)
grupo_2 = rng_r.normal(3.5, 1.0, 12)
inf = informe_comparacion(grupo_1, grupo_2)
respuesta = None    # TODO: "a", "b" o "c"

# Comprobación automática
assert inf is not None and respuesta is not None, "Completa la función y la respuesta (TODO)"
assert set(inf) == {"dif", "ic_inf", "ic_sup", "t", "p", "d"}, "el diccionario debe tener exactamente las claves dif, ic_inf, ic_sup, t, p y d"
assert np.isclose(inf["dif"], grupo_1.mean() - grupo_2.mean()), "dif = media de a − media de b"
assert np.isclose(inf["t"], -1.1355, atol=1e-3) and np.isclose(inf["p"], 0.2706, atol=1e-3), "usa Welch: stats.ttest_ind(a, b, equal_var=False)"
assert np.isclose(inf["ic_inf"], -1.4931, atol=1e-3) and np.isclose(inf["ic_sup"], 0.4440, atol=1e-3), "usa res.confidence_interval(confidence_level=0.95)"
assert np.isclose(inf["d"], -0.4636, atol=1e-3), "d = dif / desviación típica combinada"
assert respuesta == "b", "un p-value alto no demuestra que no haya efecto; mira el IC"
print(f"dif = {inf['dif']:.3f}   IC 95 % = ({inf['ic_inf']:.3f}, {inf['ic_sup']:.3f})   d = {inf['d']:.2f}   p = {inf['p']:.3f}")
print("Reto final correcto ✔")

---
## Conclusión de la práctica

Has calculado un t a mano, comparado grupos independientes (Welch) y pareados, contrastado dos proporciones, comprobado por simulación la tasa de falsos positivos y la potencia, y redactado un informe con efecto, IC y p-value. Siguiente paso: las clases 8 y 9 (ANOVA, tests no paramétricos, chi-cuadrado y comparaciones múltiples).

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
tiempos = np.array([2.7, 3.1, 2.5, 3.0, 2.8, 2.6, 3.2, 2.9, 2.4, 2.8])
mu0 = 3.0
n = len(tiempos)

media = tiempos.mean()
s = tiempos.std(ddof=1)
t_obs = (media - mu0) / (s / np.sqrt(n))
gl = n - 1
p_valor = 2 * stats.t.sf(abs(t_obs), gl)
rechaza_h0 = bool(p_valor < 0.05)

assert all(v is not None for v in (media, s, t_obs, gl, p_valor, rechaza_h0)), "Completa las líneas marcadas con TODO"
ref = stats.ttest_1samp(tiempos, mu0)
assert np.isclose(t_obs, ref.statistic), "t = (media − mu0) / (s / raíz de n), con s calculada con ddof=1"
assert gl == n - 1, "los grados de libertad son n − 1"
assert np.isclose(p_valor, ref.pvalue), "p = 2 * stats.t.sf(abs(t), gl)"
assert rechaza_h0 == (ref.pvalue < 0.05), "compara el p-value con α = 0,05"
print(f"t = {t_obs:.3f}   gl = {gl}   p = {p_valor:.4f}   ¿rechazo H0? {rechaza_h0}")
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
rng_e2 = np.random.default_rng(2026)
grupo_a = rng_e2.normal(3.0, 0.8, 30)
grupo_b = rng_e2.normal(3.5, 1.2, 35)

res = stats.ttest_ind(grupo_a, grupo_b, equal_var=False)
dif = grupo_a.mean() - grupo_b.mean()
na, nb = len(grupo_a), len(grupo_b)
sp = np.sqrt(((na - 1) * grupo_a.var(ddof=1) + (nb - 1) * grupo_b.var(ddof=1)) / (na + nb - 2))
d = dif / sp
ic = res.confidence_interval(confidence_level=0.95)
significativo = bool(res.pvalue < 0.05)

assert all(v is not None for v in (res, dif, d, ic, significativo)), "Completa las líneas marcadas con TODO"
assert np.isclose(res.statistic, -3.0697, atol=1e-3), "usa equal_var=False (Welch)"
assert np.isclose(dif, grupo_a.mean() - grupo_b.mean()), "dif = media de A − media de B"
assert np.isclose(d, -0.7266, atol=1e-3), "d = dif / desviación típica combinada, sp = raíz(((na−1)·va + (nb−1)·vb) / (na+nb−2))"
assert ic.low < dif < ic.high, "el IC debe contener la diferencia observada"
assert significativo is True or significativo == True, "compara el p-value con 0,05"
print(f"dif = {dif:.3f} s   IC 95 % = ({ic.low:.3f}, {ic.high:.3f})   d = {d:.2f}   p = {res.pvalue:.4f}")
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
rng_e3 = np.random.default_rng(77)
antes = rng_e3.lognormal(1.0, 0.5, 20)
despues = antes - 0.25 + rng_e3.normal(0, 0.25, 20)

p_pareado = stats.ttest_rel(antes, despues).pvalue
p_indep = stats.ttest_ind(antes, despues, equal_var=False).pvalue
usar_pareado = True

assert all(v is not None for v in (p_pareado, p_indep, usar_pareado)), "Completa las líneas marcadas con TODO"
assert np.isclose(p_pareado, 3.8e-5, rtol=0.05), "usa stats.ttest_rel(antes, despues)"
assert np.isclose(p_indep, 0.656, atol=0.005), "usa stats.ttest_ind(..., equal_var=False)"
assert usar_pareado == True, "cada página tiene su pareja antes/después: los datos son pareados"
print(f"p pareado = {p_pareado:.2e}   p independiente = {p_indep:.3f}")
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
conv = np.array([90, 66])
vis = np.array([1200, 1200])

z, p = proportions_ztest(conv, vis)
p1, p2 = conv / vis
dif = p1 - p2
ee = np.sqrt(p1 * (1 - p1) / vis[0] + p2 * (1 - p2) / vis[1])
ic = (dif - 1.96 * ee, dif + 1.96 * ee)
rechaza_h0 = bool(p < 0.05)

assert all(v is not None for v in (z, p, dif, ic, rechaza_h0)), "Completa las líneas marcadas con TODO"
assert np.isclose(z, 1.9872, atol=1e-3), "usa proportions_ztest(conv, vis) con la variante nueva primero"
assert np.isclose(p, 0.0469, atol=1e-3), "el p-value es el de dos colas que devuelve proportions_ztest"
assert np.isclose(dif, 0.02), "dif = 90/1200 − 66/1200"
assert ic[0] < dif < ic[1], "el IC debe contener la diferencia"
assert np.isclose(ic[0], 0.0003, atol=5e-4) and np.isclose(ic[1], 0.0397, atol=5e-4), "IC = dif ± 1,96 · raíz(p1(1−p1)/n1 + p2(1−p2)/n2)"
assert rechaza_h0 == True, "p < 0,05"
print(f"z = {z:.3f}   p = {p:.4f}   dif = {dif:.4f}   IC 95 % = ({ic[0]:.4f}, {ic[1]:.4f})")
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
rng_ej = np.random.default_rng(11)
N = 5_000

x = rng_ej.normal(0, 1, (N, 25))
y = rng_ej.normal(0, 1, (N, 25))
tasa_fp = (stats.ttest_ind(x, y, axis=1, equal_var=False).pvalue < 0.01).mean()

x2 = rng_ej.normal(0, 1, (N, 25))
y2 = rng_ej.normal(0.8, 1, (N, 25))
potencia = (stats.ttest_ind(x2, y2, axis=1, equal_var=False).pvalue < 0.05).mean()

assert tasa_fp is not None and potencia is not None, "Completa las líneas marcadas con TODO"
ee_fp = np.sqrt(0.01 * 0.99 / N)
assert abs(tasa_fp - 0.01) < 4 * ee_fp, "con H0 cierta, la tasa de p < α debe ser ≈ α (tolerancia: 4 errores estándar)"
assert abs(potencia - 0.79) < 0.04, "con efecto 0,8 y n = 25 por grupo la potencia teórica es ≈ 0,79"
print(f"tasa de falsos positivos (α = 0,01) = {tasa_fp:.4f}   potencia = {potencia:.3f}")
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def informe_comparacion(a, b):
    """Devuelve dif, IC 95 % (Welch), t, p y d de Cohen."""
    res = stats.ttest_ind(a, b, equal_var=False)
    ic = res.confidence_interval(confidence_level=0.95)
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    dif = np.mean(a) - np.mean(b)
    return {"dif": dif, "ic_inf": ic.low, "ic_sup": ic.high, "t": res.statistic, "p": res.pvalue, "d": dif / sp}

rng_r = np.random.default_rng(910)
grupo_1 = rng_r.normal(3.0, 1.0, 12)
grupo_2 = rng_r.normal(3.5, 1.0, 12)
inf = informe_comparacion(grupo_1, grupo_2)
respuesta = "b"

assert inf is not None and respuesta is not None, "Completa la función y la respuesta (TODO)"
assert set(inf) == {"dif", "ic_inf", "ic_sup", "t", "p", "d"}, "el diccionario debe tener exactamente las claves dif, ic_inf, ic_sup, t, p y d"
assert np.isclose(inf["dif"], grupo_1.mean() - grupo_2.mean()), "dif = media de a − media de b"
assert np.isclose(inf["t"], -1.1355, atol=1e-3) and np.isclose(inf["p"], 0.2706, atol=1e-3), "usa Welch: stats.ttest_ind(a, b, equal_var=False)"
assert np.isclose(inf["ic_inf"], -1.4931, atol=1e-3) and np.isclose(inf["ic_sup"], 0.4440, atol=1e-3), "usa res.confidence_interval(confidence_level=0.95)"
assert np.isclose(inf["d"], -0.4636, atol=1e-3), "d = dif / desviación típica combinada"
assert respuesta == "b", "un p-value alto no demuestra que no haya efecto; mira el IC"
print(f"dif = {inf['dif']:.3f}   IC 95 % = ({inf['ic_inf']:.3f}, {inf['ic_sup']:.3f})   d = {inf['d']:.2f}   p = {inf['p']:.3f}")
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (p-value, error de tipo I/II, potencia, pareado frente a independiente, tamaño del efecto)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*